## 뉴스 데이터


### 네이버 뉴스 크롤링


In [1]:
import datetime
import json
import re
import time
from urllib.parse import urljoin, urlparse
from zoneinfo import ZoneInfo

import requests
from bs4 import BeautifulSoup

In [2]:
# 검색 조건 설정
query = "삼성전자"
max_results = 100
sort = {"relevance": 0, "latest": 1}["relevance"]
timeout = 10
delay = 0.7

# 네이버 뉴스 검색 주소
search_url = "https://search.naver.com/search.naver"

# 일반 웹 브러우저 요청처럼 보이도록 설정
header = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
    )
}

# 여러 요청에서 동일한 헤더를 사용
session = requests.Session()
session.headers.update(header)

In [3]:
# 기사 URL과 중복 확인용 집합 초기화
news_urls = []
seen_urls = set()
page = 0

# 원하는 개수만큼 검색 결과 페이지를 순회
while len(news_urls) < max_results:
    search_params = {
        "where": "news",
        "query": query,
        "sort": sort,
        "start": page * 10 + 1,
    }

    response = session.get(
        search_url,
        params=search_params,
        timeout=timeout,
    )
    response.raise_for_status()

    search_soup = BeautifulSoup(response.text, "html.parser")
    found_this_page = 0

    # 네이버 뉴스 기사 링크만 추출
    for link in search_soup.select("a[href]"):
        href = link.get("href") or ""
        parsed = urlparse(href)

        match = re.match(r"^/mnews/article/(\d{3})/(\d+)", parsed.path)
        if parsed.netloc != "n.news.naver.com" or match is None:
            continue

        office_id, article_id = match.groups()
        article_url = f"https://n.news.naver.com/mnews/article/{office_id}/{article_id}"

        # 중복되지 않은 기사 URL만 수집
        if article_url in seen_urls:
            continue

        seen_urls.add(article_url)
        news_urls.append(article_url)
        found_this_page += 1

        if len(news_urls) >= max_results:
            break

    page += 1

    if len(news_urls) < max_results:
        time.sleep(delay)

if not news_urls:
    raise RuntimeError("네이버 뉴스 검색 결과를 찾지 못했습니다.")

print(f"수집할 기사 URL: {len(news_urls)}개")
news_urls[:3]

수집할 기사 URL: 100개


['https://n.news.naver.com/mnews/article/003/0014105279',
 'https://n.news.naver.com/mnews/article/014/0005556413',
 'https://n.news.naver.com/mnews/article/031/0001046863']

In [4]:
# 각 기사 페이지에서 제목과 본문 수집
articles = []

for index, article_url in enumerate(news_urls):
    try:
        # 기사 페이지 요청
        response = session.get(
            article_url,
            timeout=timeout,
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # 기사 제목 추출
        title_element = soup.select_one("#title_area")
        if title_element is None:
            raise RuntimeError("기사 제목 영역을 찾지 못했습니다.")

        title = " ".join(title_element.get_text(" ", strip=True).split())

        # 기사 본문 추출
        content = soup.select_one("#newsct_article")
        if content is None:
            raise RuntimeError("기사 본문 영역을 찾지 못했습니다.")

        content_text = content.get_text("\n", strip=True)

        # 기사 본문의 이미지와 캡션 추출
        images = []
        seen_image_urls = set()

        for image_element in content.select(".end_photo_org img"):
            image_url = (image_element.get("src") or "").strip()
            if not image_url:
                continue

            image_url = urljoin(article_url, image_url)
            if image_url in seen_image_urls:
                continue

            seen_image_urls.add(image_url)

            # 현재 이미지가 포함된 사진 영역 탐색
            image_container = image_element.find_parent(class_="end_photo_org")

            caption = ""

            if image_container is not None:
                caption_element = image_container.select_one(".img_desc")

                if caption_element is not None:
                    caption = caption_element.get_text(" ", strip=True)
                    caption = " ".join(caption.split())

            images.append(
                {
                    "url": image_url,
                    "caption": caption,
                }
            )

        # 기사 작성 시간 추출
        written_at_element = soup.select_one("._ARTICLE_DATE_TIME")
        if written_at_element is None:
            raise RuntimeError("기사 작성 시간 영역을 찾지 못했습니다.")

        written_at_text = (written_at_element.get("data-date-time") or "").strip()

        if not written_at_text:
            raise RuntimeError("기사 작성 시간 값을 찾지 못했습니다.")

        written_at = (
            datetime.datetime.strptime(
                written_at_text,
                "%Y-%m-%d %H:%M:%S",
            )
            .replace(tzinfo=ZoneInfo("Asia/Seoul"))
            .isoformat()
        )

        # 기사 수정 시간 추출
        modified_at_element = soup.select_one("._ARTICLE_MODIFY_DATE_TIME")

        modified_at = None

        if modified_at_element is not None:
            modified_at_text = (
                modified_at_element.get("data-modify-date-time") or ""
            ).strip()

            if modified_at_text:
                modified_at = (
                    datetime.datetime.strptime(
                        modified_at_text,
                        "%Y-%m-%d %H:%M:%S",
                    )
                    .replace(tzinfo=ZoneInfo("Asia/Seoul"))
                    .isoformat()
                )

        # 수집한 기사 저장
        articles.append(
            {
                "query": query,
                "title": title,
                "content_text": content_text,
                "images": images,
                "written_at": written_at,
                "modified_at": modified_at,
                "url": article_url,
            }
        )

    except requests.RequestException as error:
        print(f"요청 실패: {article_url}, {type(error).__name__}: {error}")

    except RuntimeError as error:
        print(f"수집 실패: {article_url}, {type(error).__name__}: {error}")

    if index + 1 < len(news_urls):
        time.sleep(delay)

print(f"본문 수집 성공: {len(articles)}개")

본문 수집 성공: 100개


In [5]:
# 첫 번째 수집 결과 확인
articles[0]

{'query': '삼성전자',
 'title': '삼성전자 소액주주들 "자사주 45.5조 매입해야" 이사회 압박',
 'content_text': '"잉여현금 50%, 배당 아닌 자사주로 즉시 집행해야"\n"마이크론·키옥시아처럼 100%로 기준 높여야"\n주주환원·성과급 관련 임시주총 소집 추진\n*재판매 및 DB 금지\n[서울=뉴시스] 박주연 기자 = 소액주주 플랫폼 액트가 삼성전자 이사회에 45조5000억원 규모의 자사주 매입·소각을 요구키로 했다.\n액트는 최근 자사주 매입·소각에 대한 플랫폼 찬반 투표에서 참여 주식 수 기준 100.0%(13만2836주)의 압도적 찬성을 확인했으며, 3일 오전부터 연대 서명을 받고 있다고 이날 밝혔다. 액트는 연대 서명을 완료한 후 이를 이사회에 정식 제출할 계획이다.\n액트는 "삼성전자는 스스로 약속한 잉여현금 50% 환원 기준에 따라 45조5000억원을 전액 자사주 매입·소각에 투입할 것을 촉구한다"며 "나아가 마이크론·키옥시아 등 글로벌 경쟁사 수준으로 환원 기준 자체를 상향할 것을 함께 요구한다"고 밝혔다.\n액트는 "현저한 저평가를 고려하면 자사주 매입 수익률은 21.7%로 배당 요구수익률 13.7%(액트 추산)를 압도한다"며 "이 확실한 수익률을 포기하고 현금을 유보하려면, 이사회는 그보다 높은 내부 투자 수익률을 주주 앞에 정량적으로 입증해야 한다"고 밝혔다.\n액트는 "마이크론은 잉여현금 100% 환원을, 키옥시아는 분기 핵심 잉여현금 8272억엔의 약 97%(8000억엔)를 자사주 매입에 투입하기로 했다"며 "삼성이 이 수준으로 상향하면 3분기에만 약 90조원의 자사주 매입이 가능하다"고 주장했다.\n이상목 액트 대표는 "경쟁사들이 잉여현금의 100% 가까이를 주주에게 돌려주는 동안 삼성전자는 환원 계획만 발표하고 현금 190조원을 곳간에 쌓아두고 있다"며 "마지막 자사주 골든타임마저 놓친다면 시장의 신뢰는 돌아오지 않을 것"이라고 지적했다.\n액트는 향후 서한의 상세 논리를 담은 자료를 언론에 공유하

In [6]:
# 수집한 뉴스를 JSONL 파일로 저장
output_path = "news.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    file.writelines(
        json.dumps(article, ensure_ascii=False) + "\n" for article in articles
    )

print(f"저장 완료: {output_path}")

저장 완료: news.jsonl
